# Stage 5 — Signed Clutch Action Values

**Pipeline position:** Stage 3 `vaep_all_leagues.parquet` + Stage 4 `leverage_index.parquet` → **Stage 5** → `clutch_values_signed.parquet` → Stage 6

---

## What this notebook does

Every on-ball action receives a value in **league points**:

```
V(aᵢ) = ΔP_scores(aᵢ) · L⁺  −  ΔP_concedes(aᵢ) · L⁻
```

- `ΔP_scores`, `ΔP_concedes` — how much the action changed the acting team's probability of scoring / conceding within the next 10 actions (Stage 3).
- `L⁺`, `L⁻` — expected league points gained if the acting team scores next / lost if it concedes next (Stage 4).

Creating a chance is worth what scoring would be worth; preventing one is worth what conceding would have cost. **One formula for every action and every position** — no special case for shots and no separate treatment by position, so values are directly comparable across the whole squad.

Values are computed for **all** actions. The late-game window is a filter applied in Stage 6, not a property of the value.

**Hard gates**
1. The two channels, recombined as `ΔP_scores − ΔP_concedes`, must reproduce Stage 3's `vaep_value` exactly.
2. Every action must find its Stage 4 leverage, and the two sources must agree on minute and window.

## Outputs

| File | Content |
|---|---|
| `data/clutch_values_signed.parquet` | Per-action channels, leverage, signed value `V`, and the two baseline values |
| `stage5_clutch/value_by_position_state.csv` | Value totals and shares by position and score state, for all three weightings |
| `stage5_clutch/value_by_channel.csv` | The same totals split into the scoring and conceding channels |
| `stage5_clutch/plots/lead_protection_share.png` | Share of each position's value earned while protecting a lead |

## 0. Setup

In [ ]:
!pip install -q pandas numpy pyarrow matplotlib

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print('Libraries loaded.')

In [ ]:
# Repository root (the notebooks live one level below it)
BASE_DIR = Path('..') if Path.cwd().name in ('pipeline-stages', 'feature-engineering') else Path('.')

VAEP_PATH    = BASE_DIR / 'stage3_vaep' / 'vaep_all_leagues.parquet'
LI_PATH      = BASE_DIR / 'data' / 'leverage_index.parquet'
PLAYERS_PATH = BASE_DIR / 'WyScout' / 'players.json'
OUTPUT_PATH  = BASE_DIR / 'data' / 'clutch_values_signed.parquet'

STAGE5_DIR = BASE_DIR / 'stage5_clutch'
PLOTS_DIR  = STAGE5_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

KEYS         = ['game_id', 'team_id', 'player_id', 'absolute_second']
SCORE_STATES = ['chasing', 'level', 'protecting']
POSITIONS    = ['forward', 'midfielder', 'defender', 'goalkeeper']

print('Configuration ready.')
for p in (VAEP_PATH, LI_PATH, PLAYERS_PATH):
    print(f"  {'OK     ' if p.exists() else 'MISSING'} {p}")
print(f'  Output:  {OUTPUT_PATH}')

## 1. Load Inputs

| Source | Columns used |
|---|---|
| Stage 3 `vaep_all_leagues.parquet` | probabilities `p_scores` / `p_concedes`, `is_goal_action`, `vaep_value` |
| Stage 4 `leverage_index.parquet` | `L_plus`, `L_minus`, `W`, the one-sided `leverage_index`, and game state |
| WyScout `players.json` | Position group — **for reporting only**; it never enters the value |

In [ ]:
VAEP_COLS = KEYS + ['league_country', 'minute', 'is_post_75', 'action_type', 'result',
                    'is_goal_action', 'p_scores', 'p_concedes', 'vaep_value']
vaep = pd.read_parquet(VAEP_PATH, columns=VAEP_COLS)
print(f"Stage 3: {len(vaep):,} actions, {vaep['game_id'].nunique():,} matches")

# The channels below compare each action with the one stored just before it,
# so every match must occupy one contiguous block of rows.
gid    = vaep['game_id'].values
blocks = int((gid[1:] != gid[:-1]).sum()) + 1
assert blocks == vaep['game_id'].nunique(), 'matches are not stored contiguously'
print(f'  {blocks:,} contiguous match blocks')

In [ ]:
LI_COLS = KEYS + ['minute', 'is_post_75', 'gameweek', 'goal_difference', 'score_state',
                  'W', 'L_plus', 'L_minus', 'leverage_index']
li = pd.read_parquet(LI_PATH, columns=LI_COLS)
print(f'Stage 4: {len(li):,} actions')
print(f"  mean L+ = {li['L_plus'].mean():.4f}   mean L- = {li['L_minus'].mean():.4f}   (league points)")

In [ ]:
ROLE_MAP = {'Goalkeeper': 'goalkeeper', 'Defender': 'defender',
            'Midfielder': 'midfielder', 'Forward': 'forward'}

with open(PLAYERS_PATH, encoding='utf-8') as f:
    players = json.load(f)

positions = pd.DataFrame({
    'player_id':      [int(p.get('wyId', p.get('playerId'))) for p in players],
    'position_group': [ROLE_MAP.get(p.get('role', {}).get('name'), 'unknown') for p in players],
}).drop_duplicates('player_id')

print(f'Positions: {len(positions):,} players')
print(positions['position_group'].value_counts().to_string())

## 2. The Two Action-Value Channels

For each action we measure how it changed the two Stage 3 probabilities.

| Situation | ΔP_scores | ΔP_concedes |
|---|---|---|
| First action of a match, or the action right after a goal | `P_scores(Sᵢ)` | `P_concedes(Sᵢ)` |
| Same team as the previous action | `P_scores(Sᵢ) − P_scores(Sᵢ₋₁)` | `P_concedes(Sᵢ) − P_concedes(Sᵢ₋₁)` |
| Possession changed | `P_scores(Sᵢ) − P_concedes(Sᵢ₋₁)` | `P_concedes(Sᵢ) − P_scores(Sᵢ₋₁)` |

A goal ends the passage of play, so the next action starts from a fresh baseline, exactly like kick-off. When possession changes, the previous state is read from the other team's side, so the two channels swap.

**Hard gate:** `ΔP_scores − ΔP_concedes` must reproduce Stage 3's `vaep_value` exactly — the channels are the same quantity, kept apart.

In [ ]:
p_s  = vaep['p_scores'].values
p_c  = vaep['p_concedes'].values
team = vaep['team_id'].values
goal = vaep['is_goal_action'].values.astype(bool)


def prev(a, fill):
    """The previous row's value (fill for the very first row)."""
    return np.concatenate([[fill], a[:-1]])


new_game = gid != prev(gid, -1)
vaep['after_goal'] = prev(goal, False) & ~new_game     # previous action, same match, was a goal
reset = new_game | vaep['after_goal'].values           # fresh baseline: kick-off or after a goal
same  = team == prev(team, -1)

p_s_prev, p_c_prev = prev(p_s, np.nan), prev(p_c, np.nan)
vaep['dP_scores']   = np.where(reset, p_s, np.where(same, p_s - p_s_prev, p_s - p_c_prev))
vaep['dP_concedes'] = np.where(reset, p_c, np.where(same, p_c - p_c_prev, p_c - p_s_prev))

# ── HARD GATE: the channels must rebuild the Stage 3 value exactly ────────────
err = np.abs((vaep['dP_scores'] - vaep['dP_concedes']) - vaep['vaep_value']).max()
print(f'max |(dP_scores - dP_concedes) - vaep_value| = {err:.2e}')
assert err < 1e-9, 'Channels do not reproduce the Stage 3 action values — stop here.'
print('Channels reproduce the Stage 3 values exactly.')

## 3. Attach the Signed Leverage from Stage 4

`(game, team, player, second)` is not unique: a player can make two actions within the same second. Both stages keep the source order, so numbering the repeats within each key gives an exact one-to-one match.

**Hard gate:** every action matches, and the two sources agree on the minute and the post-75' flag.

In [ ]:
vaep['_k'] = vaep.groupby(KEYS, sort=False).cumcount()
li['_k']   = li.groupby(KEYS, sort=False).cumcount()

df = vaep.merge(li, on=KEYS + ['_k'], how='inner', suffixes=('', '_li'), validate='one_to_one')

checks = [
    ('every action matched',     len(df) == len(vaep) == len(li),
     f'{len(df):,} of {len(vaep):,} (Stage 3) / {len(li):,} (Stage 4)'),
    ('minute agrees',             bool((df['minute'] == df['minute_li']).all()), ''),
    ("post-75' flag agrees",      bool((df['is_post_75'] == df['is_post_75_li']).all()), ''),
]
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<26} {detail}")
assert all(ok for _, ok, _ in checks), 'Stage 3 and Stage 4 rows do not line up — stop here.'

df = (df.drop(columns=['_k', 'minute_li', 'is_post_75_li'])
        .merge(positions, on='player_id', how='left'))
df['position_group'] = df['position_group'].fillna('unknown')
del vaep, li
print(f'\nMerged: {len(df):,} actions x {df.shape[1]} columns')

## 4. Signed Clutch Value

```
V = ΔP_scores · L⁺  −  ΔP_concedes · L⁻          (league points)
```

Two baselines are kept alongside it, on the same channels, so Stage 6 can isolate what each design choice adds:

| Column | Weighting | Isolates |
|---|---|---|
| `vaep_value` | none | the value of leverage weighting at all |
| `V_sym` | one-sided index: `(ΔP_scores − ΔP_concedes) · LI` | the value of weighting each side by its own stake |
| `V` | signed: `L⁺` on the scoring channel, `L⁻` on the conceding channel | — primary metric |

In [ ]:
df['V']     = df['dP_scores'] * df['L_plus'] - df['dP_concedes'] * df['L_minus']
df['V_sym'] = df['vaep_value'] * df['leverage_index']

print('Per-action values, all actions:')
print(df[['vaep_value', 'V_sym', 'V']].describe().round(5).to_string())

## 5. Validation

| Check | Why |
|---|---|
| C1 no missing values | every action is valued |
| C2 goals earn ≈ their full `L⁺` | a goal moves the score by one, so its value should be close to `(1 − P_scores before) · L⁺` |
| C3 the action after a goal is ≈ 0 on average | the goal is counted once, not again on the restart |
| C4 decided games are ≈ 0 | at 85'+ with a 3-goal margin nothing can change the result |
| C5 missed shots < 0 < goals | a wasted chance costs a little; a goal earns a lot |

In [ ]:
goals = df['is_goal_action'].astype(bool)
late  = df['minute'] >= 85
shots = df['action_type'].isin(['shot', 'shot_penalty', 'shot_freekick'])

checks = []

n_nan = int(df[['dP_scores', 'dP_concedes', 'L_plus', 'L_minus', 'V']].isna().sum().sum())
checks.append(('C1  no missing values', n_nan == 0, f'{n_nan} NaN'))

ratio = df.loc[goals, 'V'].sum() / df.loc[goals, 'L_plus'].sum()
checks.append(('C2  goals: sum V / sum L+ in [0.7, 1.0]', 0.7 <= ratio <= 1.0, f'{ratio:.3f}'))

post = df.loc[df['after_goal'], 'V'].mean()
checks.append(('C3  action after a goal: |mean V| < 0.01', abs(post) < 0.01, f'{post:+.5f}'))

dead = df.loc[late & (df['goal_difference'].abs() >= 3), 'V'].abs().mean()
checks.append(("C4  85'+ and |g| >= 3: mean |V| < 0.002", dead < 0.002, f'{dead:.5f}'))

miss, hit = df.loc[shots & ~goals, 'V'].mean(), df.loc[goals, 'V'].mean()
checks.append(('C5  missed shots < 0 < goals', miss < 0 < hit, f'miss {miss:+.4f}   goal {hit:+.4f}'))

print('=' * 84)
print('SIGNED VALUE VALIDATION')
print('=' * 84)
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<46} {detail}")
print('=' * 84)
assert all(ok for _, ok, _ in checks), 'Signed value checks failed — stop here.'

## 6. Where the Value Comes From

**By action type** — mean and total signed value, next to the unweighted value.

**By position and score state** — for each weighting, the share of a position's total value earned in each score state.

**By channel** — the same totals split into the scoring channel (`ΔP_scores`, weighted by `L⁺`) and the conceding channel (`−ΔP_concedes`, weighted by `L⁻`). This shows which part of a player's value each weight acts on. While protecting a lead `L⁺` is small, so value from building attacks is down-weighted — a further goal changes little. Value from reducing the risk of conceding is weighted by `L⁻` and kept. How much this moves a position's total therefore depends on how much of its value the action-value model places in the conceding channel.

In [ ]:
by_type = (df.groupby('action_type')
             .agg(n=('V', 'size'),
                  mean_unweighted=('vaep_value', 'mean'),
                  mean_signed=('V', 'mean'),
                  total_signed=('V', 'sum'))
             .sort_values('total_signed', ascending=False))
print('Value by action type (all actions):')
print(by_type.round(5).to_string())

In [ ]:
METRICS = {'unweighted': 'vaep_value', 'symmetric': 'V_sym', 'signed': 'V'}   # symmetric = one-sided index

pos_state = (df[df['position_group'].isin(POSITIONS)]
             .groupby(['position_group', 'score_state'])
             .agg(n=('V', 'size'), **{k: (v, 'sum') for k, v in METRICS.items()}))
totals = pos_state.groupby(level='position_group')[list(METRICS)].transform('sum')
share  = (pos_state[list(METRICS)] / totals).add_suffix('_share')
pos_state = pos_state.join(share)

print('Total value and share of each position total, by score state (all actions):')
print(pos_state.round(4).to_string())
pos_state.to_csv(STAGE5_DIR / 'value_by_position_state.csv')
print(f"\nSaved: {STAGE5_DIR / 'value_by_position_state.csv'}")

In [ ]:
CHANNELS = {'scoring':     lambda d: d['dP_scores'],
            'conceding':   lambda d: -d['dP_concedes'],
            'scoring_w':   lambda d: d['dP_scores'] * d['L_plus'],
            'conceding_w': lambda d: -d['dP_concedes'] * d['L_minus']}

by_channel = (df[df['position_group'].isin(POSITIONS)]
              .assign(**CHANNELS)
              .groupby(['position_group', 'score_state'])[list(CHANNELS)].sum())
print('Value by channel ("conceding" = -dP_concedes, so positive means the risk of conceding fell):')
print(by_channel.round(1).to_string())
by_channel.to_csv(STAGE5_DIR / 'value_by_channel.csv')

defensive = df[df['action_type'].isin(['interception', 'tackle', 'clearance', 'keeper_save'])]
print('\nDefensive actions - mean unweighted value by channel:')
print(defensive.assign(**{k: CHANNELS[k] for k in ('scoring', 'conceding')})
               .groupby('action_type')[['scoring', 'conceding']].mean().round(5).to_string())

In [ ]:
# ── Share of each outfield position's value earned while protecting a lead ────
# Three series, fixed colours (validated for colour-vision deficiency); every bar carries its label.
SERIES  = [('unweighted_share', 'Unweighted',            '#2a78d6'),
           ('symmetric_share',  'One-sided leverage',    '#eb6834'),
           ('signed_share',     'Signed leverage (L±)',  '#1baf7a')]
INK, INK_2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
OUTFIELD = ['defender', 'midfielder', 'forward']

prot = pos_state.xs('protecting', level='score_state').reindex(OUTFIELD)

fig, ax = plt.subplots(figsize=(10, 5), facecolor=SURFACE)
x, w = np.arange(len(OUTFIELD)), 0.26
for j, (col, label, colour) in enumerate(SERIES):
    xs = x + (j - 1) * (w + 0.02)
    ax.bar(xs, prot[col] * 100, w, color=colour, label=label)
    for xi, v in zip(xs, prot[col] * 100):
        ax.annotate(f'{v:.0f}%', (xi, v), xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=10, color=INK)

ax.set_xticks(x)
ax.set_xticklabels([p.title() for p in OUTFIELD], color=INK)
ax.set_ylabel('% of the position\'s total value', color=INK_2)
ax.set_title('Share of value earned while protecting a lead', loc='left', fontsize=13, color=INK)
ax.legend(frameon=False, fontsize=10, labelcolor=INK)
ax.set_facecolor(SURFACE)
ax.grid(axis='y', color=GRID, lw=1, ls='-')
ax.set_axisbelow(True)
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_color(GRID)
ax.tick_params(colors=INK_2, length=0)

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'lead_protection_share.png', dpi=150, bbox_inches='tight', facecolor=SURFACE)
plt.show()
print(f"Saved: {PLOTS_DIR / 'lead_protection_share.png'}")

## 7. Save

All actions are kept. Stage 6 applies the analysis window, the playing-time thresholds and the aggregation.

In [ ]:
OUT_COLS = KEYS + ['league_country', 'gameweek', 'minute', 'is_post_75',
                   'score_state', 'goal_difference', 'position_group',
                   'action_type', 'result', 'is_goal_action', 'after_goal',
                   'dP_scores', 'dP_concedes', 'vaep_value',
                   'W', 'L_plus', 'L_minus', 'leverage_index',
                   'V', 'V_sym']
df[OUT_COLS].to_parquet(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: ({len(df):,}, {len(OUT_COLS)})')

## 8. Next Step → Stage 6

Stage 6 aggregates `V` to players: `CPI(p) = (m / n_p) · Σ V`, with `m = 56` (median actions per appearance) and
`n_p ≥ 700`, ranked within position.